# Question 5 - 169. Majority Element

Given an array `nums` of size `n`, return *the majority element*.

The majority element is the element that appears **more than** `⌊n / 2⌋` times. You may assume that the majority element always exists in the array.

**Example 1:**

    Input: nums = [3,2,3]
    Output: 3

**Example 2:**

    Input: nums = [2,2,1,1,1,2,2]
    Output: 2

**Constraints:**

- `n == nums.length`
- `1 <= n <= 5 * 10^4`
- `-10^9 <= nums[i] <= 10^9`

**Follow-up:** Could you solve the problem in linear time and in `O(1)` space?

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Boyer–Moore voting (cancel out pairs of different values)

📘 **Revise first:** [Pattern E · counting](./0-concepts-array-and-string.ipynb) · [Part 6 · dict/Counter costs](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (count each value) | O(n²) | O(1) |
| 1½ Hash map counts | O(n) | O(n) |
| 1½ Sort, take the middle | O(n log n) | O(1)–O(n) |
| 2️⃣ Optimized (Boyer–Moore vote) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

One value appears in **more than half** of the positions. Find it.

`[2, 2, 1, 1, 1, 2, 2]` has 7 items. More than half means at least 4. `2` appears 4 times → answer `2`.

We're **promised** such a value exists, so we don't have to handle "no majority".

**Everyday picture: a battle where every soldier fights one enemy.** Imagine each number is a soldier wearing a team colour. Soldiers from **different** teams pair up and knock each other out, one for one. Since the majority team has **more soldiers than all the other teams combined**, after all the one-for-one knockouts, **someone from the majority team is always left standing.**

That's the whole idea behind the optimal algorithm (Boyer–Moore voting). We'll build up to it.

### Step 0 · Clarify before coding

🗣️ *"The majority element appears more than n/2 times, and it's guaranteed to exist, so I don't need to verify it at the end. Values can be large or negative, and n is up to 50,000."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Is a majority **guaranteed**? | **Yes.** | The vote algorithm's result can be trusted without a second pass. |
| "More than half" or "at least half"? | **Strictly more** than ⌊n/2⌋. | With exactly half (e.g. `[1,2]`) the pairing argument breaks. |
| Value range? | ±10⁹. | Too large for a "count array indexed by value". Use a dict, or something smarter. |
| Size? | Up to 5·10⁴. | O(n²) ≈ 2.5 billion steps, which is too slow. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** for each value, count how many times it appears. If the count is more than n/2, that's the answer.

🗣️ *"The most direct approach: for each element, count its occurrences by scanning the whole array, and return the first one whose count exceeds n/2."*

**Why is it slow?** Counting one value means scanning all n items. Doing that for up to n values is **n × n** work. For 50,000 items that's 2.5 billion steps.

🗣️ *"That's O(n²) because every count is a full scan. The obvious waste is that I recount the same value every time it shows up."*

In [1]:
class SolutionBrute:
    def majorityElement(self, nums: list[int]) -> int:
        n = len(nums)
        for x in nums:
            if sum(1 for y in nums if y == x) > n // 2:   # full scan per value
                return x

#### Step 1½ · Two reasonable improvements

**(a) Count everything once with a hash map** (a Python `dict`: an O(1) lookup table from value to count).
One pass to tally, then return whichever count is over n/2. **O(n) time, but O(n) extra memory** for the dict.

**(b) Sort, then take the middle element.**
If one value fills more than half the positions, then after sorting **it must cover the middle position**, wherever its block starts. **O(n log n)** for the sort.

```
sorted: [1, 1, 1, 2, 2, 2, 2]
                  ↑ middle index 3 → 2
```

🗣️ *"A hash map of counts gets me O(n) time but uses O(n) memory. Sorting and taking the middle element is neat, because the majority must cover the middle, but it's O(n log n). The follow-up asks for O(n) time and O(1) space, which neither gives me."*

In [2]:
from collections import Counter


class SolutionHashMap:
    def majorityElement(self, nums: list[int]) -> int:
        counts = Counter(nums)                  # value -> how many times
        return max(counts, key=counts.get)      # the most common value


class SolutionSort:
    def majorityElement(self, nums: list[int]) -> int:
        return sorted(nums)[len(nums) // 2]     # majority always covers the middle

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** if I **cancel out** pairs of *different* values, the majority **can't be fully cancelled**, because it has more members than everyone else combined. So I don't need to count everything. I just keep a running "survivor".

**Boyer–Moore voting**, in plain words. Keep two things:
- `candidate`: the value currently "standing"
- `count`: how many unmatched supporters it has right now

Walk through the numbers:
- If `count == 0`, nobody is standing, so the current number becomes the new `candidate`.
- If the number **equals** the candidate → `count += 1` (another supporter).
- If it's **different** → `count -= 1` (one supporter and this number knock each other out).

At the end, `candidate` is the majority.

**Why it works (the one-sentence proof):** every decrement cancels **one** majority vote with **at most one** non-majority vote. Since the majority has more than n/2 votes, it can't be cancelled to zero by fewer than n/2 others, so it's the one left at the end.

🗣️ *"I'll use Boyer–Moore voting. I keep a candidate and a counter. When the counter is zero, I adopt the current number as the candidate. If the next number matches the candidate I increment, otherwise I decrement, which is like pairing up two different values and throwing both away. Throwing away pairs of different values never changes which value is the majority, because the majority loses at most one per pair and it has more than half. So the surviving candidate at the end is the answer. One pass, two variables: O(n) time, O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Count each value by rescanning | ❌ | O(n²). Recounts the same values again and again. |
| **Hash map** of counts | ✅ good | O(n) time, very clear, but O(n) extra memory. The best answer if values could have no majority (you need real counts). |
| Sort and take the middle | ✅ neat | Short, but O(n log n), and `sorted()` makes an O(n) copy. |
| Count array indexed by value | ❌ | Values go up to ±10⁹. You can't make an array that big. |
| Bit-by-bit voting (32 bits) | ⚠️ | O(32·n) and O(1) space, works, but it's more complex and fiddly with negative numbers. |
| **Boyer–Moore vote** | ✅ **best** | O(n) time and O(1) space. Exactly the follow-up's requirement. |

**Important caveat to say out loud:** Boyer–Moore *assumes* a majority exists. If it might not, do a **second pass** to count the candidate and confirm it's > n/2.

In [3]:
class Solution:
    def majorityElement(self, nums: list[int]) -> int:
        candidate, count = None, 0
        for x in nums:
            if count == 0:          # nobody standing: x takes over
                candidate = x
            if x == candidate:
                count += 1          # another supporter
            else:
                count -= 1          # cancel one supporter against x
        return candidate

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [2, 2, 1, 1, 1, 2, 2]`

| x | count == 0? → new candidate | x == candidate? | count after | candidate |
|---|---|---|---|---|
| 2 | yes → candidate = 2 | ✅ | 1 | 2 |
| 2 | | ✅ | 2 | 2 |
| 1 | | ❌ cancel | 1 | 2 |
| 1 | | ❌ cancel | 0 | 2 |
| 1 | yes → candidate = **1** | ✅ | 1 | 1 |
| 2 | | ❌ cancel | 0 | 1 |
| 2 | yes → candidate = **2** | ✅ | 1 | **2** ✅ |

**Narrated:** "Two 2s step up, so the 2 team has 2 standing. Two 1s arrive and knock both 2s out, leaving the field empty. The third 1 steps up as the new candidate, but the next 2 knocks it out. The last 2 steps up, and it's the only one left standing. The answer is 2."

Notice the candidate **changed** in the middle and was briefly wrong (`1`). That's fine. The guarantee is only about the **final** survivor.

In [4]:
cases = [
    ([3, 2, 3], 3),
    ([2, 2, 1, 1, 1, 2, 2], 2),
    ([1], 1),
    ([-1, -1, 5], -1),
    ([6, 5, 5], 5),
    ([1, 2, 1, 2, 1], 1),        # majority by just one
]
for nums, expected in cases:
    for Impl in (SolutionBrute, SolutionHashMap, SolutionSort, Solution):
        assert Impl().majorityElement(nums) == expected, Impl.__name__
    print(nums, "->", expected)

import random
for _ in range(500):
    n = random.randint(1, 15)
    maj = random.randint(-3, 3)
    k = n // 2 + 1
    nums = [maj] * k + [random.randint(-3, 3) for _ in range(n - k)]
    random.shuffle(nums)
    assert Solution().majorityElement(nums) == maj
print("All tests passed ✅")

[3, 2, 3] -> 3
[2, 2, 1, 1, 1, 2, 2] -> 2
[1] -> 1
[-1, -1, 5] -> -1
[6, 5, 5] -> 5
[1, 2, 1, 2, 1] -> 1
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Recount each value | O(n²) | O(1) | n values × a full scan each. |
| Hash map counts | O(n) | O(n) | One pass, but stores a count per distinct value. |
| Sort + middle | O(n log n) | O(1)–O(n) | The sort dominates. |
| **Boyer–Moore** | **O(n)** | **O(1)** | One pass, two variables. |

**Why O(n) time?** A single loop with constant work per item.

**Why O(1) memory?** Just `candidate` and `count`, however long the list is.

**Edge cases to mention:** a single element, a majority by exactly one (`[1,2,1,2,1]`), negative values.

**Likely follow-ups:**
- *"What if a majority might not exist?"* → After the vote, count the candidate in a second pass and check it's > n/2. Still O(n)/O(1).
- *"Elements appearing more than n/3 times?"* (LC 229) → Keep **two** candidates and two counters. There can be at most two such values.

---

#### 🗣️ Full interview script (about 60 seconds)

*"I need the element that appears more than half the time, and it's guaranteed to exist.*

*Brute force counts each element by scanning the array, which is O(n²). A hash map of counts brings it to O(n) time but O(n) space. Sorting and taking the middle element also works, since the majority must cover the middle, but that's O(n log n).*

*For O(n) time and O(1) space I'll use Boyer–Moore voting. I keep a candidate and a count. When the count is zero I adopt the current number. Matching numbers increment the count and different numbers decrement it. Each decrement cancels one majority vote against one other vote, and since the majority has more than half, it's the one left standing at the end.*

*If the majority weren't guaranteed, I'd add a second pass to verify the candidate."*